# Open Weights Inference

This notebook contains the procedure for text generation from open weights model. The models are loaded from HuggigngFace.

In [1]:
import gc
from collections import defaultdict

import torch.cuda
from dotenv import load_dotenv

import pandas as pd
from tqdm.autonotebook import tqdm

from datasets import Dataset
from huggingface_hub import login
from transformers.pipelines.pt_utils import KeyDataset
from transformers import pipeline, GenerationConfig, AutoTokenizer, AutoModelForCausalLM

from inference.evaluator import Evaluator
from inference.batch_request_handler import BatchRequestHandler

load_dotenv()
login()

/tmp/ipykernel_17184/729122637.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Helper Functions

In [2]:
def merge_and_write(
    dataframe: pd.DataFrame,
    subset_df: pd.DataFrame,
    file_name: str,
    id_column: str = "static_id"
):
    """Merges two dataframes.

    This function updates the entries in dataframe with the corresponding values in subset_df. Any
    entries in dataframe will be overwritten.

    Args:
        dataframe (pd.DataFrame): DataFrame object. The entries in this dataframe will be overwritten.
        subset_df (pd.DataFrame): Subset dataframe that will be used to update dataframe.
        file_name (str): File name of the written dataframe.
        id_column (str): Column to use as keys for the updates.
    """
    for col in subset_df.columns:
        if col not in dataframe.columns:
            dataframe[col] = None

    dataframe.set_index(id_column, inplace=True)
    subset_df = subset_df.set_index(id_column)

    # Any value in dataframe will be overwritten by subset_df
    dataframe.update(subset_df)
    dataframe.reset_index(inplace=True)

    # df_file = f"{drive_path}/{file_name}.parquet"
    df_file = f"data/{file_name}.parquet"
    dataframe.to_parquet(df_file)

def prepare_prompt(batch, fn_tokenizer):
    prompts_list = []
    for system, user in zip(batch["persona"], batch["prompt"]):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
        prompts_list.append(
            fn_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )
    return {"prompt": prompts_list}

## Load Model

In [3]:
model_string = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_string, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    model_string,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
)
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    dtype=torch.bfloat16,
)
pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## Run Inference

In [4]:
evaluator = Evaluator()
persona_registry = evaluator.get_persona_registry()
personas = persona_registry.get_names()

2026-02-10 07:54:34  INFO     dataset_handler:_load               Preparing datasets
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Loading mmlu-pro
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Selecting a subset of 200 samples per category for category
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Dataframe length: 2800
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Loading MATH
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Selecting a subset of 200 samples per category for subject
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Dataframe length: 500
2026-02-10 07:54:34  INFO     dataset_handler:_download_dataset   Loading flores
2026-02-10 07:54:35  INFO     dataset_handler:_download_dataset   Selecting a subset of 200 samples per category for iso_639_3
2026-02-10 07:54:35  INFO     dataset_handler:_download_dataset   Dataframe length: 41400


In [5]:
for ds_name in evaluator.get_dataset_names():
    if ds_name == "flores":
        continue
    print(f" === {ds_name} ===")

    # Load dataset
    _, dataset_df = evaluator.get_data(ds_name)
    qs_type = evaluator.get_question_type(ds_name)
    dataset_df = dataset_df.copy()

    # Bulk create question prompts
    prompts = dataset_df.apply(
        BatchRequestHandler.create_question_prompt, axis=1,
        question_type=qs_type)
    dataset_df.loc[:, "prompt"] = prompts

    # Sort the prompts by length to minimize padding overhead
    prompts_tokenized = pipe.tokenizer(prompts.to_list())
    dataset_df.loc[:, "length"] = [len(p) for p in prompts_tokenized["input_ids"]]
    dataset_df = dataset_df.sort_values(by="length", ascending=False)

    # Long format allows using HF's batch inference pipeline
    long_df = pd.melt(
        dataset_df,
        id_vars=["static_id", "prompt"],
        value_vars=personas,
        var_name="persona_col",
        value_name="persona"
    )
    dataset_hf = Dataset.from_pandas(long_df)

    # Apply the chat template here
    dataset_hf = dataset_hf.map(
        prepare_prompt,
        batched=True,
        fn_kwargs={"fn_tokenizer": tokenizer},
        num_proc=4
    )

    gen_cfg = GenerationConfig(
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id
    )
    for batch_size in [8, 4, 3, 2, 1]:
        print(f"Testing batch size {batch_size}")
        try:
            responses = defaultdict(list)
            # noinspection PyTypeChecker
            for static_id, persona, out in tqdm(
                zip(
                    long_df["static_id"],
                    long_df["persona_col"],
                    pipe(
                        KeyDataset(dataset_hf, "prompt"),
                        batch_size=batch_size, return_full_text=False,
                        generation_config=gen_cfg
                    )
                ), total=len(dataset_hf)
            ):
                responses["static_id"].append(static_id)
                responses["persona"].append(persona.replace("persona", "answer"))
                responses["completion"].append(out[0]["generated_text"])

            # pivot back from long to wide
            response_df = pd.DataFrame(responses)
            response_df = response_df.pivot(index="static_id", columns="persona", values="completion")
            response_df = response_df.reset_index()

            # merge and save
            merge_and_write(dataset_df, response_df, f"{ds_name}_model")
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()


 === mmlu-pro ===


Map (num_proc=4):   0%|          | 0/12000 [00:00<?, ? examples/s]

Testing batch size 8


  0%|          | 0/12000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 === MATH ===


Map (num_proc=4):   0%|          | 0/4210 [00:00<?, ? examples/s]

Testing batch size 8


  0%|          | 0/4210 [00:00<?, ?it/s]

In [ ]:
# currently testing max_token = 2048. Let's see whether it works!